## 🚀 Map Berhasil Dibuat!

### 📋 Summary Data:
* **695 devices** with positions (687 online, 5 offline, 3 unknown)
* **474 geofence polygons** displayed
* **Area**: Kalimantan region (centered at -3.6°, 115.65°)

### 🎮 Cara Menggunakan Map:

**Layer Control (kanan atas)**:
* 🟢 **Online Devices** - Toggle on/off cluster hijau
* 🔴 **Offline Devices** - Toggle on/off cluster merah  
* 🟦 **Geofences** - Toggle on/off polygon biru
* **Base Map** - Switch antara OpenStreetMap ↔️ Satellite

**Interactive Features**:
* **Zoom**: Scroll wheel atau tombol +/-
* **Pan**: Drag map untuk geser
* **Click Cluster**: Otomatis zoom ke area cluster
* **Click Marker**: Popup info device (nama, status, speed, koordinat)
* **Click Polygon**: Popup info geofence (nama, ID, deskripsi)
* **Hover**: Tooltip cepat tanpa klik
* **Fullscreen**: Klik ikon di kiri atas

### ✨ Features yang Sudah Dikonfigurasi:
1. **Marker Clustering** - Untuk performa optimal dengan 695+ devices
2. **Color-coded Status**:
   - Hijau (🟢) = Device online
   - Merah (🔴) = Device offline
   - Orange (🟠) = Status unknown
3. **Geofence Overlay** - 474 polygon dengan styling semi-transparent
4. **Dual Base Map** - OpenStreetMap dan Satellite imagery
5. **Rich Popups** - Info lengkap per device/geofence

---

**⚠️ Note:** Map menggunakan Folium (bukan Kepler.gl) karena compatibility dengan Databricks environment

# Traccar Fleet Map Visualization with Kepler.gl

Visualisasi geospasial untuk 900+ devices dengan:
* **Device positions** - Latest position per device dengan status (online/offline)
* **Geofence overlays** - 1,428 geofence polygon areas
* Interactive map dengan hover info

In [0]:
# Kepler.gl may have compatibility issues, so we'll use Folium as alternative
# Folium works well in Databricks and handles large datasets better
# %pip install folium

In [0]:
# Get latest position per device with device info
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Get latest position per device
window_spec = Window.partitionBy('deviceid').orderBy(F.col('servertime').desc())

device_positions_df = spark.sql("""
WITH latest_positions AS (
  SELECT 
    p.deviceid,
    p.latitude,
    p.longitude,
    p.speed,
    p.servertime,
    p.address,
    ROW_NUMBER() OVER (PARTITION BY p.deviceid ORDER BY p.servertime DESC) as rn
  FROM tc.default.tc_positions p
)
SELECT 
  d.id as device_id,
  d.name as device_name,
  d.status,
  d.category,
  lp.latitude,
  lp.longitude,
  lp.speed,
  lp.servertime as last_update,
  lp.address,
  CASE 
    WHEN d.status = 'online' THEN '#00FF00'
    WHEN d.status = 'offline' THEN '#FF0000'
    ELSE '#FFFF00'
  END as color_hex
FROM tc.default.tc_devices d
LEFT JOIN latest_positions lp ON d.id = lp.deviceid AND lp.rn = 1
WHERE lp.latitude IS NOT NULL AND lp.longitude IS NOT NULL
""")

# Convert to Pandas for Kepler.gl
device_positions_pdf = device_positions_df.toPandas()

print(f"Total devices with positions: {len(device_positions_pdf)}")
print(f"\nStatus distribution:")
print(device_positions_pdf['status'].value_counts())

display(device_positions_pdf.head())

In [0]:
import re
import json

# IMPORTANT: Geofence coordinates are stored across multiple columns!
# Need to reconstruct full WKT POLYGON string
geofences_df = spark.sql("""
SELECT 
  id,
  name,
  description,
  -- Reconstruct full POLYGON by concatenating all coordinate columns
  CONCAT(
    area,
    COALESCE(attributes, ''),
    COALESCE(calendarid, ''),
    COALESCE(geotype, ''),
    COALESCE(group_name, ''),
    '))'  -- Close the POLYGON
  ) as full_polygon
FROM tc.default.tc_geofences
WHERE area IS NOT NULL
  AND area LIKE 'POLYGON%'
""")

geofences_pdf = geofences_df.toPandas()

print(f"Total geofences found: {len(geofences_pdf)}")

# Function to parse WKT POLYGON to GeoJSON coordinates
def parse_wkt_polygon(wkt_string):
    try:
        # Extract coordinates from POLYGON ((lat lon, lat lon, ...))
        wkt_clean = wkt_string.replace('POLYGON ((', '').replace('))', '')
        
        # Split into coordinate pairs (separated by space then comma)
        # Format: "lat1 lon1 lat2 lon2 lat3 lon3" -> need to group by pairs
        coords_str = wkt_clean.strip()
        
        # Split by whitespace to get all numbers
        numbers = coords_str.split()
        
        coordinates = []
        # Group into pairs: [lon, lat] for GeoJSON format
        for i in range(0, len(numbers), 2):
            if i+1 < len(numbers):
                lat = float(numbers[i])
                lon = float(numbers[i+1])
                coordinates.append([lon, lat])  # GeoJSON format: [lon, lat]
        
        # Close the polygon if not already closed
        if len(coordinates) > 2:
            if coordinates[0] != coordinates[-1]:
                coordinates.append(coordinates[0])
            return coordinates
        else:
            return None
            
    except Exception as e:
        print(f"Error parsing: {e}")
        return None

# Parse polygons
geofences_pdf['coordinates'] = geofences_pdf['full_polygon'].apply(parse_wkt_polygon)

# Filter out failed parses
geofences_pdf = geofences_pdf[geofences_pdf['coordinates'].notna()].copy()

print(f"Successfully parsed {len(geofences_pdf)} geofence polygons")
print(f"\nSample polygon (first 3 coordinates):")
if len(geofences_pdf) > 0:
    sample_coords = geofences_pdf.iloc[0]['coordinates'][:3]
    print(f"  {geofences_pdf.iloc[0]['name']}: {sample_coords}")
    print(f"  Total coordinates: {len(geofences_pdf.iloc[0]['coordinates'])}")
    display(geofences_pdf[['id', 'name', 'description']].head())

In [0]:
# Convert geofences to GeoJSON FeatureCollection format
geojson_features = []

for idx, row in geofences_pdf.iterrows():
    feature = {
        "type": "Feature",
        "properties": {
            "id": int(row['id']),
            "name": row['name'],
            "description": row['description'] if row['description'] else ""
        },
        "geometry": {
            "type": "Polygon",
            "coordinates": [row['coordinates']]  # Note: array of arrays for GeoJSON
        }
    }
    geojson_features.append(feature)

geofence_geojson = {
    "type": "FeatureCollection",
    "features": geojson_features
}

print(f"Created GeoJSON with {len(geojson_features)} geofence polygons")
print(f"\nSample feature structure:")
print(json.dumps(geojson_features[0], indent=2) if geojson_features else "No features")

In [0]:
import folium
from folium.plugins import MarkerCluster
import branca.colormap as cm

# Create base map centered on Kalimantan area
map_center = [-3.6, 115.65]
m = folium.Map(
    location=map_center,
    zoom_start=10,
    tiles='OpenStreetMap',  # Can change to 'Stamen Terrain', 'CartoDB positron', etc.
    width='100%',
    height='800px'
)

# Add satellite imagery option
folium.TileLayer('Esri.WorldImagery', name='Satellite').add_to(m)

print(f"Map created centered at {map_center}")
print(f"Ready to add {len(device_positions_pdf)} devices and {len(geojson_features)} geofences")

In [0]:
# Create marker clusters for better performance with many devices
marker_cluster_online = MarkerCluster(name='Online Devices').add_to(m)
marker_cluster_offline = MarkerCluster(name='Offline Devices').add_to(m)

# Add device markers
for idx, row in device_positions_pdf.iterrows():
    # Determine marker color based on status
    if row['status'] == 'online':
        color = 'green'
        cluster = marker_cluster_online
        icon = 'play'
    elif row['status'] == 'offline':
        color = 'red'
        cluster = marker_cluster_offline
        icon = 'stop'
    else:
        color = 'orange'
        cluster = marker_cluster_online
        icon = 'question'
    
    # Create popup with device info
    popup_html = f"""
    <b>Device:</b> {row['device_name']}<br>
    <b>Status:</b> <span style='color:{color};font-weight:bold'>{row['status']}</span><br>
    <b>Category:</b> {row['category']}<br>
    <b>Speed:</b> {row['speed']:.1f} km/h<br>
    <b>Last Update:</b> {row['last_update']}<br>
    <b>Coordinates:</b> {row['latitude']:.6f}, {row['longitude']:.6f}
    """
    
    # Add marker
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{row['device_name']} ({row['status']})",
        icon=folium.Icon(color=color, icon=icon, prefix='fa')
    ).add_to(cluster)

print(f"Added {len(device_positions_pdf)} device markers with clustering")

In [0]:
# Add geofence polygons to map
geofence_layer = folium.FeatureGroup(name='Geofences', show=True).add_to(m)

for feature in geojson_features:
    # Extract coordinates
    coords = feature['geometry']['coordinates'][0]
    
    # Convert to Folium format [lat, lon]
    folium_coords = [[lat, lon] for lon, lat in coords]
    
    # Create popup
    popup_html = f"""
    <b>Geofence:</b> {feature['properties']['name']}<br>
    <b>ID:</b> {feature['properties']['id']}<br>
    <b>Description:</b> {feature['properties']['description']}
    """
    
    # Add polygon
    folium.Polygon(
        locations=folium_coords,
        color='#0066CC',      # Dark blue border
        fill=True,
        fillColor='#00D4FF',  # Cyan fill
        fillOpacity=0.3,
        weight=2,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=feature['properties']['name']
    ).add_to(geofence_layer)

print(f"Added {len(geojson_features)} geofence polygons")

In [0]:
# Add layer control to toggle layers on/off
folium.LayerControl(collapsed=False).add_to(m)

# Add fullscreen button
from folium.plugins import Fullscreen
Fullscreen(position='topleft').add_to(m)

# Display the map
m